# Tokenization
The purpose of tokenizers is to translate text into data, so that a llm or AI model can process it.

## Types
Character-level
Chunk-level
    Byte-pair

- BPE
- BLT
- H-Net


# References
https://www.youtube.com/watch?v=zduSFxRajkE&t=1800s

In [1]:
import torch
import tiktoken
import regex as re

In [5]:
comp = re.compile(r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s""")
print(re.findall(comp, "Hello world123 how're you"))

['Hello', ' world', '123', ' how', "'re", ' you']


In [9]:
with open("../data/input.txt") as f:
    text = f.read()

#print(text)

In [ ]:
#text = "The Tokenizer is a necessary and pervasive component of Large Language Models (LLMs), where it translates between strings and tokens (text chunks). Tokenizers are a completely separate stage of the LLM pipeline: they have their own training sets, training algorithms (Byte Pair Encoding), and after training implement two fundamental functions: encode() from strings to tokens, and decode() back from tokens to strings. In this lecture we build from scratch the Tokenizer used in the GPT series from OpenAI. In the process, we will see that a lot of weird behaviors and problems of LLMs actually trace back to tokenization. We'll go through a number of these issues, discuss why tokenization is at fault, and why someone out there ideally finds a way to delete this stage entirely."

#text = text.encode("utf-8")
#tokens = re.findall(comp, text)
tokens = list(text.encode("utf-8"))

print(len(tokens))

263198


In [4]:
text = "This is some text"
byte_array = bytearray(text, "utf-8")
ids = list(byte_array)
print(ids)

print(f"# of characters: {len(text)}")
print(f"# of token IDs: {len(ids)}")

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]
# of characters: 17
# of token IDs: 17


In [11]:
def get_stats(ids):
    count = {}

    for i in range(len(ids) - 1):
        try:
            pair = ids[i], ids[i+1]
            count[pair] = count.get(pair, 0) + 1
        except Exception as e:
            print("Error :", e)

    return count


stats = get_stats(tokens)
#print(stats)
print(sorted(((v, k) for k, v in stats.items()), reverse=True))

[(1150, (',', ' and')), (798, (',\n', 'And')), (780, (',', ' I')), (745, (':\n', 'I')), (523, (',', ' my')), (432, ('\n', 'To')), (412, (',', ' sir')), (388, (';\n', 'And')), (354, (',', ' that')), (345, (' to', ' the')), (325, (';', ' and')), (317, ('.\n\n', 'KING')), (315, ('\n', 'And')), (305, (' in', ' the')), (297, (',', ' to')), (297, (' you', ',')), (284, (' of', ' the')), (283, (':\n', 'And')), (277, (',\n', 'That')), (276, (':\n', 'What')), (267, (':\n', 'O')), (263, (',', ' as')), (259, (' I', "'ll")), (258, ('\n', 'The')), (256, (' I', ' have')), (254, (':\n', 'The')), (253, (',', ' the')), (250, ('\n', 'That')), (249, (',\n', 'To')), (249, (',\n', 'I')), (247, (' my', ' lord')), (246, (',', ' but')), (246, (' I', ' am')), (238, (',', ' for')), (236, ('KING', ' RICHARD')), (236, (',', ' if')), (222, (' me', ',')), (221, (' sir', ',')), (220, (',', ' you')), (218, (':\n', 'My')), (213, ('.\n\n', 'DUKE')), (213, ("'s", 't')), (211, ('GLOUCESTER', ':\n')), (208, (' to', ' be'))

In [14]:
top_pair = max(stats, key=stats.get)
print(f"Top Pair: {top_pair}")
print(chr(32), chr(116))

Top Pair: (',', ' and')
  t


In [15]:
def merge(ids, pair, idx):
    i = 0
    new_ids = []
    while i < len(ids):
        x, y = pair
        if i < (len(ids) - 1) and x == ids[i] and y == ids[i+1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids

#print(merge([5,6,7,8,8,1], (7,8), 259))
#print(merge(tokens, top_pair, 256))

In [17]:

ids = list(tokens)

def train(ids):
    vocab_size = 276
    num_merges = vocab_size - 256

    merges = {}
    for i in range(num_merges):
        stats = get_stats(ids)
        top_pair = max(stats, key=stats.get)
        idx = 256 + i
        print(f"Merging {top_pair} into a new token {idx}")
        ids = merge(ids, top_pair, idx)
        merges[top_pair] = idx

    return ids, merges

ids, merges = train(list(tokens))
print(len(list(tokens)))
print(len(ids))

Merging (',', ' and') into a new token 256
Merging (',\n', 'And') into a new token 257
Merging (',', ' I') into a new token 258
Merging (':\n', 'I') into a new token 259
Merging (',', ' my') into a new token 260
Merging ('\n', 'To') into a new token 261
Merging (',', ' sir') into a new token 262
Merging (';\n', 'And') into a new token 263
Merging (',', ' that') into a new token 264
Merging (' to', ' the') into a new token 265
Merging (';', ' and') into a new token 266
Merging ('.\n\n', 'KING') into a new token 267
Merging ('\n', 'And') into a new token 268
Merging (' in', ' the') into a new token 269
Merging (' of', ' the') into a new token 270
Merging (':\n', 'And') into a new token 271
Merging (',', ' to') into a new token 272
Merging (',\n', 'That') into a new token 273
Merging (':\n', 'What') into a new token 274
Merging (':\n', 'O') into a new token 275
263198
254342


In [18]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (x, y), m in merges.items():
    #print(vocab[x] + vocab[y])
    vocab[m] = vocab[x] + vocab[y]

def decode(ids: list):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

decode([128])

KeyError: ','

In [94]:
print(stats)
print(merges)

{(70, 105): 308, (105, 114): 2700, (114, 115): 2649, (115, 116): 6823, (116, 32): 16508, (32, 67): 693, (67, 105): 122, (105, 116): 6114, (116, 105): 2977, (105, 122): 208, (122, 101): 285, (101, 110): 7568, (110, 58): 587, (58, 10): 8762, (10, 66): 1669, (66, 101): 367, (101, 102): 903, (102, 111): 3325, (111, 114): 8458, (114, 101): 9843, (101, 32): 27643, (32, 119): 10546, (119, 101): 2982, (32, 112): 4490, (112, 114): 1720, (114, 111): 3613, (111, 99): 373, (99, 101): 3354, (101, 101): 3807, (101, 100): 3676, (100, 32): 14165, (32, 97): 13541, (97, 110): 10197, (110, 121): 484, (121, 32): 10283, (32, 102): 6563, (102, 117): 563, (117, 114): 5313, (114, 116): 2278, (116, 104): 22739, (104, 101): 18203, (101, 114): 11771, (114, 44): 1540, (44, 32): 14098, (32, 104): 11925, (101, 97): 6288, (97, 114): 7081, (114, 32): 10516, (32, 109): 10786, (109, 101): 6135, (32, 115): 12287, (115, 112): 1136, (112, 101): 1989, (97, 107): 1561, (107, 46): 148, (46, 10): 6676, (10, 10): 7223, (10, 65

In [ ]:
def encode(text: str):
    tokens = list(text.encode("utf-8"))

    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges: 
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)

    return tokens

print(encode("Hello workld"))


[72, 101, 275, 269, 119, 268, 107, 108, 100]


In [96]:
print(decode(encode("hello world")))

hello world


In [ ]:
def save_vocab():
    pass

def save_merges():
    pass

In [104]:
import regex as re

#comp = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}++| ?\p{N}++| ?[^\s\p{L}\p{N}]++|\s++$|\s+(?!\S)|\s""")
#comp = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?[\p{L}]+| ?[\p{N}]+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
comp = re.compile(r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s""")
print(re.findall(comp, "Hello world123 how're you"))

['Hello', ' world', '123', ' how', "'re", ' you']
